# Koschei Sentinel — Kaggle Micro-First Cyber SFT

This is the preferred free-GPU path. It first performs a real QLoRA update on pinned Qwen3.5-0.8B-Base, verifies and attests the resulting adapter, then evaluates whether the same runtime is allowed to attempt Qwen3.5-9B-Base.

Set **Accelerator = GPU** and **Internet = ON**. The 9B phase is deliberately disabled by default to protect free GPU quota.

In [ ]:
import subprocess
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
from pathlib import Path
import subprocess

repo = Path('/kaggle/working/koschei-sentinel')
if not (repo / '.git').is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/bugsbuny243/koschei-sentinel.git', str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', 'main'], check=True)
repo_commit = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
print('Repo commit:', repo_commit)


## Phase 1 — real Qwen3.5-0.8B micro-smoke

This performs actual optimizer steps. It is pipeline proof, not a production model.

In [ ]:
import os, subprocess

micro_env = os.environ.copy()
micro_env['KOSCHEI_KAGGLE_OUTPUT_ROOT'] = '/kaggle/working/koschei-sentinel-micro-output'
micro_env['HF_HOME'] = '/kaggle/working/hf-cache'
subprocess.run(
    ['bash', str(repo / 'scripts/run_cyber_sft_qwen35_08b_micro_kaggle.sh'), str(repo)],
    check=True,
    env=micro_env,
)


In [ ]:
import json
from pathlib import Path

micro_out = Path('/kaggle/working/koschei-sentinel-micro-output')
micro_receipt = json.loads((micro_out / 'run' / 'training-receipt.json').read_text())
micro_verify = json.loads((micro_out / 'export-verification.json').read_text())
scale_gate = json.loads((micro_out / 'scale-gate.json').read_text())

assert micro_receipt['global_step'] > 0
assert micro_verify['valid'] is True

print('Micro global step:', micro_receipt['global_step'])
print('GPU:', micro_receipt['cuda_device_name'])
print('GPU VRAM GiB:', round(micro_receipt['cuda_total_memory_gb'], 3))
print('Micro peak allocated GiB:', round(micro_receipt['max_cuda_memory_allocated_gb'], 3))
print('Micro peak reserved GiB:', round(micro_receipt['max_cuda_memory_reserved_gb'], 3))
print('9B disposition:', scale_gate['disposition'])
print('9B allowed:', scale_gate['allowed_to_attempt_9b'])
print('Recipe parity:', {
    'quantization_bits_match': scale_gate['quantization_bits_match'],
    'compute_dtype_match': scale_gate['compute_dtype_match'],
    'lora_target_coverage': scale_gate['lora_target_coverage'],
    'micro_context': scale_gate['micro_max_sequence_length'],
    'target_context': scale_gate['target_max_sequence_length'],
})
print('Warnings:', scale_gate['warnings'] or 'none')
print('Blockers:', scale_gate['blockers'] or 'none')
print('Micro ZIP: /kaggle/working/koschei-sentinel-qwen35-08b-micro.zip')


## Phase 2 — optional 9B smoke

Leave `RUN_9B = False` unless the scale gate above says `allowed_to_attempt_9b=true` and you want to spend additional Kaggle GPU quota. The 9B launcher independently re-verifies the micro bundle before doing any 9B work.

In [ ]:
RUN_9B = False  # Change to True only after reviewing scale_gate above.

if RUN_9B:
    if not scale_gate['allowed_to_attempt_9b']:
        raise RuntimeError('Micro scale gate blocks the 9B run on this runtime')
    nine_env = os.environ.copy()
    nine_env['KOSCHEI_KAGGLE_OUTPUT_ROOT'] = '/kaggle/working/koschei-sentinel-output'
    nine_env['KOSCHEI_MICRO_EXPORT_ROOT'] = str(micro_out)
    nine_env['KOSCHEI_KAGGLE_PROFILE'] = 'auto'
    nine_env['HF_HOME'] = '/kaggle/working/hf-cache'
    subprocess.run(
        ['bash', str(repo / 'scripts/run_cyber_sft_qwen35_9b_kaggle.sh'), str(repo)],
        check=True,
        env=nine_env,
    )
    print('9B phase completed.')
else:
    print('9B phase intentionally skipped. Micro evidence and ZIP are preserved.')


In [ ]:
nine_out = Path('/kaggle/working/koschei-sentinel-output')
if (nine_out / 'export-verification.json').is_file():
    nine_verify = json.loads((nine_out / 'export-verification.json').read_text())
    nine_receipt = json.loads((nine_out / 'run' / 'training-receipt.json').read_text())
    print('9B export valid:', nine_verify['valid'])
    print('9B global step:', nine_receipt['global_step'])
    print('9B GPU:', nine_receipt['cuda_device_name'])
    print('9B peak allocated GiB:', round(nine_receipt['max_cuda_memory_allocated_gb'], 3))
    print('9B peak reserved GiB:', round(nine_receipt['max_cuda_memory_reserved_gb'], 3))
    print('9B ZIP: /kaggle/working/koschei-sentinel-qwen35-9b-smoke.zip')
else:
    print('No 9B artifact in this session.')
